Load anomaly_scores.csv
        │
        ▼
Filter only anomalies
        │
        ▼
For each anomaly
        │
        ▼
Collect nearby logs
        │
        ▼
Collect structured metadata
        │
        ▼
Build JSON Context
        │
        ▼
Save anomaly_context.json

# Context Builder

This notebook builds contextual information around detected anomalies.

Instead of analysing an anomalous log in isolation, nearby logs and structured metadata are collected to provide additional context.

The generated context will later be passed to Gemini for automated Root Cause Analysis (RCA) and remediation recommendations.

In [1]:
import pandas as pd
import json

from pathlib import Path

In [10]:
# ==========================================
# Load Original Logs
# ==========================================

logs = pd.read_csv("../data/processed/processed_logs.csv")

# ==========================================
# Load AutoEncoder Results
# ==========================================

scores = pd.read_csv("../data/processed/anomaly_scores.csv")

print(logs.shape)
print(scores.shape)

(4945, 23)
(4945, 791)


In [15]:
logs["AnomalyScore"] = scores["AnomalyScore"]
logs["IsAnomaly"] = scores["IsAnomaly"]

df = logs.copy()

print(df.shape)

(4945, 25)


In [20]:
# ==========================================
# Filter Only Anomalous Logs
# ==========================================

anomalies = df[df["IsAnomaly"]].copy()

print(f"Total anomalies detected: {len(anomalies)}")

anomalies.head()

Total anomalies detected: 248


,Raw_Log,Component,Line_Number,Timestamp,Severity,Username,AccountID,Originating_Address,Destination_Address,TPS,...,RouterSent,ContainsError,Message_Length,Word_Count,Hour,Minute,Second,Millisecond,AnomalyScore,IsAnomaly
42,[Globals.cpp|04136|02:46:17:101|~CR~][77d27ccb...,Globals.cpp,4136,1900-01-01 02:46:17.101,~CR~,0,0.0,0,0,0.0,...,1,0,1341,135,2,46,17,101,0.914535,True
51,[Globals.cpp|01908|02:46:17:465|~CR~][8afe86c1...,Globals.cpp,1908,1900-01-01 02:46:17.465,~CR~,0,0.0,Betika! DA:255794119579 Sent to Client,255794119579,0.0,...,0,0,88,5,2,46,17,465,0.857280,True
52,[Globals.cpp|01816|02:46:17:496|~CR~][8afe86aa...,Globals.cpp,1816,1900-01-01 02:46:17.496,~CR~,0,0.0,0,0,0.0,...,0,0,1585,148,2,46,17,496,0.970583,True
53,[Globals.cpp|01908|02:46:17:496|~CR~][8afe86aa...,Globals.cpp,1908,1900-01-01 02:46:17.496,~CR~,0,0.0,GePG DA:255662337616 Sent to Client,255662337616,0.0,...,0,0,85,5,2,46,17,496,0.769156,True
86,[Globals.cpp|04136|02:46:17:590|~CR~][77d27ce5...,Globals.cpp,4136,1900-01-01 02:46:17.590,~CR~,0,0.0,0,0,0.0,...,1,0,1369,135,2,46,17,590,0.785040,True


In [16]:
# ==========================================
# Context Window Size
# ==========================================

WINDOW = 5

In [17]:
print(df.columns.tolist())

['Raw_Log', 'Component', 'Line_Number', 'Timestamp', 'Severity', 'Username', 'AccountID', 'Originating_Address', 'Destination_Address', 'TPS', 'Queue_Count', 'Packet_Number', 'AuthenticationSuccess', 'PositiveResponse', 'QueueInserted', 'RouterSent', 'ContainsError', 'Message_Length', 'Word_Count', 'Hour', 'Minute', 'Second', 'Millisecond', 'AnomalyScore', 'IsAnomaly']


In [18]:
contexts = []

for idx in anomalies.index:

    start = max(0, idx - WINDOW)

    end = min(len(df), idx + WINDOW + 1)

    window = df.iloc[start:end]

    context = {

        "LogIndex": int(idx),

        "AnomalyScore": float(df.loc[idx, "AnomalyScore"]),

        "CurrentLog": df.loc[idx, "Raw_Log"],
    

        "Component": df.loc[idx, "Component"],

        "Severity": df.loc[idx, "Severity"],

        "ContainsError": bool(df.loc[idx, "ContainsError"]),

        "AuthenticationSuccess": bool(df.loc[idx, "AuthenticationSuccess"]),

        "NearbyLogs": window["Raw_Log"].tolist()

    }

    contexts.append(context)

Total anomalies detected: 248


,Raw_Log,Component,Line_Number,Timestamp,Severity,Username,AccountID,Originating_Address,Destination_Address,TPS,...,RouterSent,ContainsError,Message_Length,Word_Count,Hour,Minute,Second,Millisecond,AnomalyScore,IsAnomaly
42,[Globals.cpp|04136|02:46:17:101|~CR~][77d27ccb...,Globals.cpp,4136,1900-01-01 02:46:17.101,~CR~,0,0.0,0,0,0.0,...,1,0,1341,135,2,46,17,101,0.914535,True
51,[Globals.cpp|01908|02:46:17:465|~CR~][8afe86c1...,Globals.cpp,1908,1900-01-01 02:46:17.465,~CR~,0,0.0,Betika! DA:255794119579 Sent to Client,255794119579,0.0,...,0,0,88,5,2,46,17,465,0.857280,True
52,[Globals.cpp|01816|02:46:17:496|~CR~][8afe86aa...,Globals.cpp,1816,1900-01-01 02:46:17.496,~CR~,0,0.0,0,0,0.0,...,0,0,1585,148,2,46,17,496,0.970583,True
53,[Globals.cpp|01908|02:46:17:496|~CR~][8afe86aa...,Globals.cpp,1908,1900-01-01 02:46:17.496,~CR~,0,0.0,GePG DA:255662337616 Sent to Client,255662337616,0.0,...,0,0,85,5,2,46,17,496,0.769156,True
86,[Globals.cpp|04136|02:46:17:590|~CR~][77d27ce5...,Globals.cpp,4136,1900-01-01 02:46:17.590,~CR~,0,0.0,0,0,0.0,...,1,0,1369,135,2,46,17,590,0.785040,True


In [21]:
import json

print(json.dumps(contexts[0], indent=4))

{
    "LogIndex": 42,
    "AnomalyScore": 0.91453516,
    "CurrentLog": "[Globals.cpp|04136|02:46:17:101|~CR~][77d27ccb]Sent Byte to Router:smi: 2010283211 teleservice_id: 4098 originating_address: \"15099\" destination_address: \"255655592461\" message_id: 138 message_type: SMS_DELIVER msg_encode: 0 user_data_type: 0 user_data_length: 157 user_data: \"ZAWADI KEMKEM! Simujanja, Saajanja, Router na Tablet, bila kusahau pesa taslim hadi Tsh 8,000,000. Tuma neno BOX kwenda 15099 kisha chagua sanduku la bahati.\" user_response_code: 255 time: \"2026/05/16 02:46:17\" validity_period: 1778975177 delivery_time: \"2026/05/16 02:46:17\" schedule_time: \"2026/05/16 02:46:17\" priority_indicator: 3 privacy_indicator: 0 reply_option: 0 vms_message_number: 0 message_alert: 255 language_indicator: 1 display_number_type: 255 retry_count: 0 status: \"R\" final_message_status: 255 destination_type: \"C\" protocol_id: 0 user_id: \"942\" taskid: \"\" msgtype: RETRY_MSG dest_imsi: \"\" orig_imsi: \"\" ori

In [22]:
# ==========================================
# Save Context
# ==========================================

from pathlib import Path
import json

Path("../data/context").mkdir(
    parents=True,
    exist_ok=True
)

with open(
    "../data/context/anomaly_context.json",
    "w"
) as f:

    json.dump(
        contexts,
        f,
        indent=4
    )

print("✅ Context saved successfully!")

✅ Context saved successfully!


In [23]:
print(len(contexts))

print(contexts[0].keys())

248
dict_keys(['LogIndex', 'AnomalyScore', 'CurrentLog', 'Component', 'Severity', 'ContainsError', 'AuthenticationSuccess', 'NearbyLogs'])
